# Fun-CosyVoice3-0.5B-2512 声音克隆（Google Colab）

1. 菜单 **代码执行程序 → 更改运行时类型**，硬件加速选 **T4 GPU**（或更好）。
2. 依次运行下面的单元格。需要更新仓库时，单独再跑 **「同步最新代码」** 格（不必重装依赖）。首次会下载约 7GB 权重，大约 10–20 分钟。
3. Colab **不启动 Gradio、不弹网页**。在「合成」格里直接跑 `demo_clone.run_clone`，`print` 和报错会出现在该格输出。改文本后重新跑合成格即可（已加载的模型会复用）。

若安装依赖时报 `Failed to build grpcio`：先 **重新运行「克隆/安装」那一格**（会 `git pull` 到已跳过 grpcio/deepspeed 的脚本）。仍异常时：**代码执行程序 → 断开连接并删除运行时**，再从头跑。

免费 Colab 磁盘和会话会过期，断开后需重新下载模型。Colab 当前是 Python 3.13，不能按官方钉死的旧 wheel 安装。

In [ ]:
!nvidia-smi -L || echo "未检测到 GPU，请先把运行时改成 T4 GPU"

**同步最新代码**（随时可再跑）。会 `git fetch` + `git pull --ff-only` 到 `main`。拉完请 **重启内核** 再跑安装/合成，以免用到内存里的旧模块。

In [ ]:
import os
os.chdir("/content")
if not os.path.isdir("/content/XGVocieClone/.git"):
    !git clone https://github.com/Mr-Yoje/XGVocieClone.git
os.chdir("/content/XGVocieClone")
!git fetch origin
!git checkout main
!git pull --ff-only origin main
!git log -1 --oneline
!chmod +x colab_webui.sh setup_colab.sh 2>/dev/null || true
print("代码已同步到上面的 commit。")

In [ ]:
import os
os.chdir("/content")
if not os.path.isdir("/content/XGVocieClone/.git"):
    !git clone https://github.com/Mr-Yoje/XGVocieClone.git
os.chdir("/content/XGVocieClone")
!git pull --ff-only || true
!bash setup_colab.sh

**纯文本 TTS**（官方默认音色）。本格在前台跑推理，日志和 traceback 直接打在下面。不要加 `--fp16`。首次会加载模型，可能要几分钟。

In [ ]:
import os
import sys

os.chdir("/content/XGVocieClone")
sys.path.insert(0, "/content/XGVocieClone")

from demo_clone import DEFAULT_TTS_TEXT, build_parser, run_clone

TTS_TEXT = DEFAULT_TTS_TEXT  # 改这里即可，重新跑本格

argv = [
    "--tts-only",
    "--force-gpu",
    "--text",
    TTS_TEXT,
    "--out-name",
    "colab_tts.wav",
]
print("开始合成（主线程，日志在本格）", flush=True)
out = run_clone(build_parser().parse_args(argv))
print("完成:", out, flush=True)

**声音克隆**：先把参考 wav 放到 Colab（例如 `/content/ref.wav`），填写转写和新文本。同样在本格前台推理，无网页。

In [ ]:
import os
import sys

os.chdir("/content/XGVocieClone")
sys.path.insert(0, "/content/XGVocieClone")

from demo_clone import DEFAULT_PROMPT_TEXT, DEFAULT_TTS_TEXT, build_parser, run_clone

PROMPT_WAV = "/content/ref.wav"  # 改成你的参考音频路径
PROMPT_TEXT = DEFAULT_PROMPT_TEXT  # 必须与录音逐字一致
TTS_TEXT = DEFAULT_TTS_TEXT

argv = [
    "--force-gpu",
    "--prompt-wav",
    PROMPT_WAV,
    "--prompt-text",
    PROMPT_TEXT,
    "--text",
    TTS_TEXT,
    "--out-name",
    "colab_clone.wav",
]
print("开始克隆（主线程，日志在本格）", flush=True)
out = run_clone(build_parser().parse_args(argv))
print("完成:", out, flush=True)

本地电脑仍可用 `python webui_clone.py` 开 Gradio。Colab 不需要停止端口；中断合成格即可停下当前推理。

In [ ]:
print("Colab 已改为笔记本内直接合成，无需 colab_webui.sh。")